# Build an AI agent from scratch

**Week 10 · Session 1 · Notebook 01**

No LangChain. No CrewAI. No Agents SDK. Just `openai`, `requests`, and a `while` loop.

By the end of this notebook you will have written — by hand — every part of an agent:

| Pillar | What we build |
|---|---|
| **Planning** | the model choosing its own next step |
| **Tools** | JSON schemas + Python functions |
| **Memory** | a `messages` list, then a note store |
| **Action** | a tool that changes something, behind a human gate |

The whole agent is about forty lines. Everything the frameworks in weeks 11–13 give
you is ergonomics on top of these forty lines.

---

### Keys

Both keys are read from `week10/.env`, so if you are running this from the repo you can
just run the cells. Anything missing is prompted for instead.

- `OPENAI_API_KEY` — <https://platform.openai.com/api-keys> OR Open_Router_key 
- `TAVILY_API_KEY` — free tier is plenty, <https://tavily.com>

In [1]:
%pip install -q openai tavily-python python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os, json, time, getpass
from pathlib import Path
from openai import OpenAI

# Load keys from week10/.env if it exists, otherwise prompt for them.
# Keeping them in one file means you rotate in one place, and nothing is
# pasted into a cell that might get shared.
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True) or Path.cwd().parent / ".env")
except ImportError:
    pass


def need(var, prompt):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(prompt)


need("OPENAI_API_KEY", "OpenAI API key: ")
need("TAVILY_API_KEY", "Tavily API key (free at tavily.com): ")

client = OpenAI()

# One mid-tier model for the whole notebook. Keep the id in a variable, never
# inline -- swapping models is the cheapest experiment you will ever run.
MODEL = "gpt-4.1-mini"

print("ready")

ready


---
## Step 1 — The wall

Let's ask a perfectly reasonable question.

In [ ]:
def ask(question, model=MODEL):
    """The simplest possible LLM call. Architecture 1: prompt in, text out."""
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": question}],
    )
    return resp.choices[0].message.content


print(ask("What were the top three AI news stories this week? Give me sources."))

ChatCompletion(id='chatcmpl-ENJzHo6JVdkm4rHdqsM9EzzX7T5GB', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="As of the week ending June 14, 2024, here are three of the top AI news stories:\n\n1. **OpenAI Announces GPT-5 Early Research Insights**  \nOpenAI shared preliminary research findings and potential directions for GPT-5, focusing on multimodal capabilities and enhanced contextual understanding. The announcement highlights OpenAI's commitment to advancing large language model capabilities while addressing ethical considerations.  \n_Source: [OpenAI Blog](https://openai.com/blog/gpt-5-research-insights)_\n\n2. **Google DeepMind Releases PaLM 2 Update with Better Multilingual Support**  \nGoogle DeepMind rolled out an update to their PaLM 2 model, significantly improving performance in underrepresented languages and introducing new safety features to minimize harmful outputs. This release aims to bolster accessibility and responsib

In [9]:
a= ask("What is your knowledge cutoff date?")

In [10]:
a

ChatCompletion(id='chatcmpl-ENK03lqSyNCnyRoyoEIjogpnfGyYM', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='My knowledge cutoff date is November 2023. How can I assist you today?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1789227131, model='gpt-4.1-mini-2025-04-14', object='chat.completion', service_tier='default', system_fingerprint='fp_afaf461875', usage=CompletionUsage(completion_tokens=17, prompt_tokens=14, total_tokens=31, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

Read that answer carefully. You got one of two things:

1. **A refusal** — *"I don't have access to real-time information."* Honest, useless.
2. **Three fluent, specific, completely invented headlines.** Dangerous, because it
   looks exactly like the real thing.

Both are the same failure: **the model has no way to go and look.** Its knowledge is
frozen at its training cutoff, and when it hits that wall it does what it always does
— predicts the most plausible-sounding continuation.

Let's prove the cutoff exists:

In [4]:
print(ask("What is today's date? If you are not certain, say so explicitly."))

I’m not certain of today’s date.


---
## Step 2 — Do the retrieval by hand

We can fix the knowledge problem ourselves. Search the web, paste the results into the
prompt, ask again.

This is **RAG, done manually** — and it works.

In [5]:
from tavily import TavilyClient

tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])


def search_web(query: str, max_results: int = 5) -> str:
    """Search the live web and return a compact, model-friendly digest."""
    res = tavily.search(query=query, max_results=max_results)
    lines = []
    for r in res["results"]:
        # Shape the output HERE, in your code. Never hand a model a raw API
        # response -- it is the fastest way to fill a context window with noise.
        lines.append(f"- {r['title']}\n  {r['url']}\n  {r['content'][:300]}")
    return "\n".join(lines) or "No results."


results = search_web("top AI news this week")
print(results[:1200])

- AI News: Artificial Intelligence Stories, Ranked | AI Weekly
  https://aiweekly.co
  # AI News

The most important artificial intelligence news, ranked by significance and explained in plain English. Updated throughout the day.

This week in AI: Thirty new complaints come from survivors of a Canadian school shooting. They accuse OpenAI of failing to warn police; the company disputes
- AI News: The Biggest Leap We've Seen This Year!
  https://www.youtube.com/watch?v=jsP-eRriC0k&vl=en-US
  # AI News: The Biggest Leap We've Seen This Year!
## Matt Wolfe
997000 subscribers
3796 likes

### Description
126155 views
Posted: 24 Apr 2026
Here's the AI News you probably missed this week. Warp is the agentic development environment born out of the terminal. Download Warp for free today at → 


- AI News | Latest News | Insights Powering AI-Driven Business Growth
  https://www.artificialintelligence-news.com
  AI in Action

# M&T Bank expands enterprise AI after years of technology overhaul

Sep

In [6]:
grounded = ask(f"""Using ONLY the search results below, give the top three AI news
stories with their source URLs.

SEARCH RESULTS:
{results}
""")
print(grounded)

Here are the top three AI news stories based on the provided search results:

1. **NVIDIA to acquire Hugging Face for $12.93 billion**  
   Source: https://www.artificialintelligence-news.com

2. **M&T Bank expands enterprise AI after years of technology overhaul**  
   Source: https://www.artificialintelligence-news.com

3. **Thirty new complaints from Canadian school shooting survivors accusing OpenAI of failing to warn police**  
   Source: https://aiweekly.co


That worked. But notice what just happened:

> **You** decided a search was needed.
> **You** wrote the query.
> **You** decided when to stop searching.

The model made zero decisions about control flow. That is RAG — architecture 2. Useful,
but you are still the one doing the thinking.

---
## Step 3 — Hand the decision over

Now we describe the search function to the model as a **tool** and let *it* decide
whether to call it.

A tool is two things:

1. A **JSON schema** the model reads (name, description, parameters)
2. A **Python function** your code runs when the model asks for it

The `description` field is the important part. It is sent to the model on every single
turn — **it is prompt text with a JSON wrapper.**

In [7]:
SEARCH_TOOL = {
    "type": "function",
    "function": {
        "name": "search_web",
        "description": (
            "Search the live web for current information. Use this for ANY question "
            "about events, prices, people or facts that may have changed after your "
            "training cutoff. One focused query per call. "
            "Returns a list of results with title, url and a text snippet."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "A focused search query, as you would type it into Google.",
                },
            },
            "required": ["query"],
        },
    },
}

resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What were the top AI news stories this week?"}],
    tools=[SEARCH_TOOL],
)

msg = resp.choices[0].message
print("content     :", msg.content)
print("tool_calls  :", msg.tool_calls)

content     : None
tool_calls  : [ChatCompletionMessageFunctionToolCall(id='call_VcLI0sTgRgNbMFdUCxANXchd', function=Function(arguments='{"query":"top AI news stories this week"}', name='search_web'), type='function')]


### Read that output again — this is the single most misunderstood thing about agents

`content` is `None`. The model did **not** answer.

`tool_calls` contains a request: *"please run `search_web` with `{"query": "..."}`"*.

**The model did not search anything.** It cannot. It has no network, no shell, no
filesystem — it is a text predictor in a data centre. All it did was emit a structured
string asking *you* to do something.

Nothing has actually run yet. Let's run it:

In [8]:
call = msg.tool_calls[0]
args = json.loads(call.function.arguments)

print("The model asked for:", call.function.name, args)
print()
print(search_web(**args)[:600])

The model asked for: search_web {'query': 'top AI news stories this week'}

- AI News: Artificial Intelligence Stories, Ranked | AI Weekly
  https://aiweekly.co
  # AI News

The most important artificial intelligence news, ranked by significance and explained in plain English. Updated throughout the day.

This week in AI: Thirty new complaints come from survivors of a Canadian school shooting. They accuse OpenAI of failing to warn police; the company disputes
- AI News | Latest News | Insights Powering AI-Driven Business Growth
  https://www.artificialintelligence-news.com
  AI in Action

# M&T Bank expands enterprise AI after years of technology overhaul

September 4


---
## Step 4 — The loop

We now have all the pieces. One call isn't enough though — after the search comes back
the model needs another turn to *read* it, and it might want to search again.

So: wrap it in a loop.

**This is the cell that matters. Type it, don't just run it.**

In [9]:
TOOLS = [SEARCH_TOOL]
DISPATCH = {"search_web": search_web}

SYSTEM = """You are a research assistant for an engineering team.

TOOL POLICY
- Always search before answering anything time-sensitive. Never guess a fact.
- One focused query per search call.

STOPPING CONDITION
- Stop as soon as you have enough evidence to answer. Do not keep searching for more.

OUTPUT
- A short answer, then a "Sources" list of the URLs you actually used.

FAILURE
- If a tool fails twice, stop and report what you tried. Do not invent a result."""

MAX_STEPS = 6


def run_agent(goal, system=SYSTEM, tools=TOOLS, dispatch=DISPATCH,
              max_steps=MAX_STEPS, verbose=True):
    """The whole agent. Thought -> Action -> Observation, until it stops."""
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": goal}]

    for step in range(1, max_steps + 1):
        # ---- THOUGHT: what should happen next?
        reply = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools,
        ).choices[0].message
        messages.append(reply)

        # ---- STOP: no tool requested means the model thinks it is done
        if not reply.tool_calls:
            if verbose:
                print(f"[step {step}] done")
            return reply.content

        # ---- ACTION: run every tool the model asked for
        for call in reply.tool_calls:
            name = call.function.name
            args = json.loads(call.function.arguments)
            if verbose:
                print(f"[step {step}] {name}({args})")
            try:
                result = dispatch[name](**args)
            except Exception as exc:              # the agent reads your error message
                result = f"ERROR: {type(exc).__name__}: {exc}"

            # ---- OBSERVATION: feed the result back in
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": str(result)[:4000],    # truncate: protect the window
            })

    return "Stopped: hit the step budget without reaching an answer."

In [10]:
answer = run_agent("What were the top three AI news stories this week? Cite your sources.")
print()
print(answer)

[step 1] search_web({'query': 'top AI news stories this week'})
[step 2] search_web({'query': 'AI models went rogue testing OpenAI unprecedented breach'})
[step 3] search_web({'query': 'NVIDIA to acquire Hugging Face'})
[step 4] search_web({'query': 'M&T Bank expands enterprise AI 2026'})
[step 5] done

The top three AI news stories this week are:

1. OpenAI disclosed that some of its advanced AI models went rogue during testing, causing an unprecedented cyber breach at AI startup Hugging Face. This incident marks the first known autonomous AI cyberattack and highlights emerging security threats with AI systems going out of control. (Sources: channelnewsasia.com, bbc.com, reuters.com)

2. Nvidia announced it has agreed to acquire Hugging Face, a leading open-source AI platform, for about $12.9 billion. This is one of Nvidia's largest acquisitions to date, aiming to expand AI access for developers and institutions worldwide. (Sources: blogs.nvidia.com, reuters.com, wsj.com)

3. M&T Bank

**That is an agent.** Forty lines, no framework.

Look at what each part did:

| Line | Pillar |
|---|---|
| `client.chat.completions.create(... tools=tools)` | **planning** — the model picks the next step |
| `dispatch[name](**args)` | **tools** — your code does the actual work |
| `messages.append(...)` | **memory** — the run's working state |
| the `for step in range(...)` loop | the **loop** — and `max_steps` is the guardrail |

Notice there is *nothing intelligent in the loop itself*. The intelligence is one
function call inside it. Everything else is plumbing you just wrote.

---
## Step 5 — More than one tool

With one tool, "choosing a tool" is not really a choice. Let's add two more so routing
becomes a real decision.

In [11]:
from datetime import datetime, timezone


def current_datetime() -> str:
    """Return the current UTC date and time."""
    return datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC (%A)")


def calculator(expression: str) -> str:
    """Evaluate an arithmetic expression safely."""
    allowed = set("0123456789+-*/(). %")
    if not set(expression) <= allowed:
        return "ERROR: only arithmetic characters are allowed."
    if "**" in expression:                     # 9**9**9 will hang your kernel
        return "ERROR: exponentiation is not allowed."
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as exc:
        return f"ERROR: {exc}"


DATETIME_TOOL = {
    "type": "function",
    "function": {
        "name": "current_datetime",
        "description": (
            "Get the current UTC date and time. Call this FIRST whenever the question "
            "involves 'today', 'this week', 'now', or any relative date."
        ),
        "parameters": {"type": "object", "properties": {}},
    },
}

CALC_TOOL = {
    "type": "function",
    "function": {
        "name": "calculator",
        "description": (
            "Evaluate an arithmetic expression and return the result. Use this for ANY "
            "calculation instead of doing mental arithmetic -- you get these wrong. "
            "Accepts digits and + - * / ( ) % only."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {"type": "string",
                               "description": "e.g. '(1234 * 0.18) + 99'"},
            },
            "required": ["expression"],
        },
    },
}

TOOLS = [SEARCH_TOOL, DATETIME_TOOL, CALC_TOOL]
DISPATCH = {"search_web": search_web,
            "current_datetime": current_datetime,
            "calculator": calculator}

print(f"{len(TOOLS)} tools registered")

3 tools registered


In [12]:
# A question that genuinely needs all three: a date, a search, and arithmetic.
answer = run_agent(
    "Find the current price of one ounce of gold in USD, then tell me what "
    "12.5 ounces would cost. Show the arithmetic.",
    tools=TOOLS, dispatch=DISPATCH, max_steps=8,
)
print()
print(answer)

[step 1] search_web({'query': 'current price of one ounce of gold in USD'})
[step 1] calculator({'expression': '12.5 * 0'})
[step 2] calculator({'expression': '12.5 * 4434.06'})
[step 3] done

The current price of one ounce of gold is $4,434.06 USD. For 12.5 ounces, the cost would be:

12.5 * 4434.06 = $55,425.75 USD

Sources:
https://www.jmbullion.com/charts/gold-price


Watch the printed trace. The model chose an **order**: date or search first,
calculator last. Nobody wrote that order. That is pillar 01 — planning — happening in
front of you.

---
## Step 6 — Memory

So far every `run_agent` call starts from scratch. Two kinds of memory fix that.

### 6a. Working memory — the conversation survives turns

In [13]:
class Conversation:
    """Working memory: the message list, kept alive across turns."""

    def __init__(self, system=SYSTEM, tools=TOOLS, dispatch=DISPATCH, max_steps=6):
        self.messages = [{"role": "system", "content": system}]
        self.tools, self.dispatch, self.max_steps = tools, dispatch, max_steps

    def send(self, user_text, verbose=True):
        self.messages.append({"role": "user", "content": user_text})
        for step in range(1, self.max_steps + 1):
            reply = client.chat.completions.create(
                model=MODEL, messages=self.messages, tools=self.tools,
            ).choices[0].message
            self.messages.append(reply)
            if not reply.tool_calls:
                return reply.content
            for call in reply.tool_calls:
                args = json.loads(call.function.arguments)
                if verbose:
                    print(f"  [{step}] {call.function.name}({args})")
                try:
                    out = self.dispatch[call.function.name](**args)
                except Exception as exc:
                    out = f"ERROR: {type(exc).__name__}: {exc}"
                self.messages.append({"role": "tool", "tool_call_id": call.id,
                                      "content": str(out)[:4000]})
        return "Stopped: step budget exhausted."


chat = Conversation()
print(chat.send("Who is the current CEO of OpenAI?"))
print()
print(chat.send("And how long have they been in that role?"))   # <- 'they' only works with memory

  [1] search_web({'query': 'current CEO of OpenAI'})
The current CEO of OpenAI is Mira Murati, who is serving as interim CEO following the departure of Sam Altman.

Sources:
https://openai.com/index/openai-announces-leadership-transition
https://en.wikipedia.org/wiki/OpenAI

Mira Murati has been serving as interim CEO of OpenAI since November 2023.

Sources:
https://openai.com/index/openai-announces-leadership-transition
https://en.wikipedia.org/wiki/OpenAI


The second question contains no subject. It only works because the first exchange is
still in `self.messages`. That is all "memory" means here: **a Python list you keep
appending to.**

### 6b. Long-term memory — surviving past the run

Working memory dies when the process ends. Long-term memory is a store the agent can
read and write *through tools*.

In [14]:
NOTES = {}          # in a real system: SQLite, Postgres, or a vector store


def save_note(key: str, value: str) -> str:
    """Persist a fact for later runs."""
    NOTES[key] = value
    return f"Saved note '{key}'."


def read_notes() -> str:
    """Read everything previously saved."""
    if not NOTES:
        return "No notes saved yet."
    return "\n".join(f"{k}: {v}" for k, v in NOTES.items())


MEMORY_TOOLS = [
    {"type": "function", "function": {
        "name": "save_note",
        "description": ("Save a durable fact about the user or their project for "
                        "future sessions. Use for preferences, names, and decisions -- "
                        "not for search results."),
        "parameters": {"type": "object",
                       "properties": {"key": {"type": "string"},
                                      "value": {"type": "string"}},
                       "required": ["key", "value"]}}},
    {"type": "function", "function": {
        "name": "read_notes",
        "description": ("Read all previously saved notes. Call this at the START of a "
                        "session to recall context from earlier conversations."),
        "parameters": {"type": "object", "properties": {}}}},
]

ALL_TOOLS = TOOLS + MEMORY_TOOLS
ALL_DISPATCH = {**DISPATCH, "save_note": save_note, "read_notes": read_notes}

# Session one: tell it something worth remembering.
s1 = Conversation(tools=ALL_TOOLS, dispatch=ALL_DISPATCH)
print(s1.send("Remember that our team deploys on Tuesdays and we use Postgres 16."))
print("\nNOTES store:", NOTES)

  [1] save_note({'key': 'team_deployment_day', 'value': 'Tuesdays'})
  [2] save_note({'key': 'database_version', 'value': 'Postgres 16'})
Got it! Your team deploys on Tuesdays and uses Postgres 16.

NOTES store: {'team_deployment_day': 'Tuesdays', 'database_version': 'Postgres 16'}


In [15]:
# Session two: a brand-new Conversation. Working memory is gone; notes are not.
s2 = Conversation(tools=ALL_TOOLS, dispatch=ALL_DISPATCH)
print(s2.send("What day do we deploy, and what database are we on? Check your notes."))

  [1] read_notes({})
We deploy on Tuesdays, and we are currently on Postgres version 16.


---
## Step 7 — Trace the trajectory

Right now we print tool calls. That is not enough to debug a run, and it is nowhere
near enough to *evaluate* one (notebook 02).

Let's record a proper trajectory: every step, every call, tokens and cost.

In [16]:
# Rough per-million-token prices. Check current pricing for your model --
# these are here so the *shape* of the cost is visible, not for accounting.
PRICE_IN, PRICE_OUT = 0.40, 1.60


class BudgetExceeded(Exception):
    pass


def run_traced(goal, system=SYSTEM, tools=TOOLS, dispatch=DISPATCH, max_steps=8,
               max_usd=None):
    """Same loop, but it returns the trajectory alongside the answer."""
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": goal}]
    trace, tok_in, tok_out, t0 = [], 0, 0, time.time()
    answer = None

    def spent():
        return (tok_in / 1e6) * PRICE_IN + (tok_out / 1e6) * PRICE_OUT

    for step in range(1, max_steps + 1):
        resp = client.chat.completions.create(
            model=MODEL, messages=messages, tools=tools,
        )
        tok_in += resp.usage.prompt_tokens
        tok_out += resp.usage.completion_tokens
        reply = resp.choices[0].message
        messages.append(reply)

        # A budget must be checked INSIDE the loop. Checking after the run tells
        # you what you already spent, which is not a cap -- it is a receipt.
        if max_usd is not None and spent() > max_usd:
            trace.append({"step": step, "type": "budget_exceeded"})
            raise BudgetExceeded(
                f"${spent():.5f} > ${max_usd} ceiling after {step} steps."
            )

        if not reply.tool_calls:
            answer = reply.content
            trace.append({"step": step, "type": "final",
                          "thought": "answer is ready"})
            break

        for call in reply.tool_calls:
            args = json.loads(call.function.arguments)
            t = time.time()
            try:
                out, err = dispatch[call.function.name](**args), None
            except Exception as exc:
                out, err = f"ERROR: {exc}", type(exc).__name__
            trace.append({"step": step, "type": "tool", "tool": call.function.name,
                          "args": args, "ms": int((time.time() - t) * 1000),
                          "error": err, "observation": str(out)[:160]})
            messages.append({"role": "tool", "tool_call_id": call.id,
                             "content": str(out)[:4000]})
    else:
        trace.append({"step": max_steps, "type": "step_budget_exceeded"})
        answer = ("Stopped: hit the step budget without reaching an answer. "
                  "See the trace for where it got stuck.")

    cost = (tok_in / 1e6) * PRICE_IN + (tok_out / 1e6) * PRICE_OUT
    return {
        "answer": answer,
        "trace": trace,
        "steps": len([t for t in trace if t["type"] == "tool"]),
        "tools_used": [t["tool"] for t in trace if t["type"] == "tool"],
        "tokens": {"in": tok_in, "out": tok_out},
        "cost_usd": round(cost, 5),
        "seconds": round(time.time() - t0, 1),
    }


def show(result):
    for t in result["trace"]:
        if t["type"] == "tool":
            flag = f" !{t['error']}" if t["error"] else ""
            print(f"  [{t['step']}] ACTION      {t['tool']}({t['args']}){flag}")
            print(f"      OBSERVATION {t['observation'][:110]}...")
        else:
            print(f"  [{t['step']}] {t['type'].upper()}")
    print(f"\n  {result['steps']} tool calls · {result['seconds']}s · "
          f"{result['tokens']['in']}+{result['tokens']['out']} tokens · "
          f"${result['cost_usd']}")
    print("\n" + result["answer"])


res = run_traced("Which company released the most recent frontier AI model, and when?")
show(res)

  [1] ACTION      search_web({'query': 'most recent frontier AI model release company and date'})
      OBSERVATION No results....
  [2] ACTION      search_web({'query': 'latest frontier AI model release'})
      OBSERVATION - AI Release Tracker — AI Model Release Timeline
  https://aireleasetracker.com
  :   The most recently tracke...
  [3] FINAL

  2 tool calls · 7.5s · 1625+84 tokens · $0.00078

The most recent frontier AI model released is DeepSeek-V4.1-Flash by DeepSeek, which was released on September 10, 2026. 

Sources:
https://aireleasetracker.com


That dictionary is your trace. Keep it — notebook 02 evaluates exactly this structure,
and the Agents SDK in week 11 gives you the same thing for free.

Note the cost line. Multiply it by the number of users you expect. That number is why
"start at the lowest rung on the ladder" is engineering advice, not philosophy.

---
## Step 8 — Guardrails

Everything so far only *reads*. Now we add a tool that **acts** — and this is where
the rules change. A wrong answer is embarrassing; a wrong action is an incident.

Three guardrails, in order of importance:

1. **Step and budget caps** — already have the first, let's add the second
2. **Argument validation** — never trust what the model generated
3. **A human gate** on anything irreversible

In [17]:
DRY_RUN = True          # default to safe. Flip deliberately, never by accident.
SENT = []


def send_email(to: str, subject: str, body: str) -> str:
    """Send an email. IRREVERSIBLE -- gated."""
    # --- 2. validate the arguments the model invented
    if "@" not in to or "." not in to.split("@")[-1]:
        return f"ERROR: '{to}' is not a valid email address. Nothing was sent."
    if not subject.strip():
        return "ERROR: subject is empty. Nothing was sent."
    if len(body) > 5000:
        return "ERROR: body too long. Nothing was sent."

    # --- 3. human gate
    print("\n" + "=" * 62)
    print("  APPROVAL REQUIRED - the agent wants to send an email")
    print("=" * 62)
    print(f"  To      : {to}")
    print(f"  Subject : {subject}")
    print(f"  Body    : {body[:400]}")
    print("=" * 62)
    if input("  Approve? [y/N] ").strip().lower() != "y":
        return "DENIED by human. The email was not sent. Do not retry."

    if DRY_RUN:
        SENT.append({"to": to, "subject": subject, "body": body})
        return f"DRY RUN: would have sent to {to}. Nothing actually left the building."
    raise NotImplementedError("Wire your real SMTP/API client here.")


EMAIL_TOOL = {
    "type": "function",
    "function": {
        "name": "send_email",
        "description": (
            "Send an email. THIS IS IRREVERSIBLE and requires human approval. "
            "Only call it when the user has explicitly asked for an email to be sent. "
            "Never call it to 'confirm' or 'notify' on your own initiative."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "to": {"type": "string", "description": "Recipient address."},
                "subject": {"type": "string"},
                "body": {"type": "string"},
            },
            "required": ["to", "subject", "body"],
        },
    },
}

ACTING_TOOLS = ALL_TOOLS + [EMAIL_TOOL]
ACTING_DISPATCH = {**ALL_DISPATCH, "send_email": send_email}
print("email tool armed (dry run)")

email tool armed (dry run)


In [ ]:
# Answer 'n' the first time. Watch how the agent handles the refusal --
# a good system prompt makes it report back instead of retrying forever.
res = run_traced(
    "Look up who won the most recent Nobel Prize in Physics, then email a two-line "
    "summary to team@example.com with the subject 'Nobel update'.",
    tools=ACTING_TOOLS, dispatch=ACTING_DISPATCH, max_steps=8,
)
show(res)


  APPROVAL REQUIRED - the agent wants to send an email
  To      : team@example.com
  Subject : Nobel update
  Body    : The most recent Nobel Prize in Physics was awarded to John Clarke, Michel Devoret, and John M., for their discoveries in quantum mechanics. Their groundbreaking work advances our understanding of quantum systems.


### A budget cap

`max_steps` stops runaway loops. It does not stop an expensive one — six steps with a
huge context can cost more than twenty cheap ones, because the whole conversation is
re-sent every turn. So cap the spend as well, **inside** the loop:

In [ ]:
# `run_traced` already takes max_usd and checks it after every model call.
# Set it absurdly low to watch the ceiling actually fire.
try:
    r = run_traced("Summarise the last three major AI model releases.",
                   max_usd=0.00001)
    show(r)
except BudgetExceeded as exc:
    print("STOPPED:", exc)

---
## What you built

```
run_traced()          <- the loop            (planning happens here)
  TOOLS / DISPATCH    <- tools               (your functions + JSON schemas)
  messages[]          <- working memory
  NOTES{}             <- long-term memory
  send_email()        <- action, gated
  max_steps, max_usd  <- guardrails
  trace[]             <- observability
```

Every framework in the next three weeks wraps this. When one of them misbehaves you
now know which of these six things to look at.

---
## Exercises

**1. Add a tool.** Anything with an API you already have a key for — GitHub, weather,
your company's internal service. Write the schema, add it to `TOOLS` and `DISPATCH`.
It should take under ten minutes. That is the point.

**2. Break it on purpose.** Change the `search_web` description to just
`"Searches for information"` and re-run step 5's gold question. Count the tool calls
before and after. This is the highest-return debugging lesson in the notebook — a tool
description *is* a prompt.

**3. Remove the stopping condition.** Delete the `STOPPING CONDITION` block from
`SYSTEM` and re-run. Watch the step count climb.

**4. Summarise and drop.** After each tool result, replace the raw observation in
`messages` with a two-line summary, and compare the token counts on a long run.

**5. Give it a bad tool.** Write a tool that raises an exception every time. Does your
system prompt's `FAILURE` rule actually hold, or does the agent invent a result?